# VGGT + Multi-View Novel View Inpainting Pipeline

This notebook runs a full pipeline with **two real input images** from `/data/` (e.g. `image_01.png`, `image_02.png`) and then augments them with an inpainted novel view:

1. Imports and setup
2. Parameters / config
3. Load and run **VGGT** on the first 2 images in `/data/`
4. Plot the initial 3D reconstruction
5. Reproject the 3D model from a VGGT camera and from a shifted camera
6. Build a hole mask from the shifted-camera projection
7. Inpaint holes with an **FLUX.2-klein local editing** model
8. Run VGGT again with the two originals + inpainted novel view and plot the new 3D model

`utils.py` contains the reusable functions used here (and was smoke-tested before this notebook was generated).


In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import utils

plt.rcParams['figure.dpi'] = 120


In [ ]:
# Parameters / config
cfg = utils.PipelineConfig(
    data_dir='data',
    output_dir='outputs/notebook_run',
    preprocess_mode='crop',
    vggt_conf_percentile=55.0,
    max_points_plot=30000,
    max_points_render=140000,
    render_point_radius=2,  # denser splats -> smaller inpainting holes
    novel_shift_right=0.10,
    novel_yaw_deg=-3.5,
    novel_pitch_deg=0.0,
    novel_roll_deg=0.0,
    mask_dilate_px=4,
    mask_close_px=3,
    inpaint_model_id='black-forest-labs/FLUX.2-klein-4B',
    inpaint_steps=6,
    inpaint_guidance_scale=1.0,
    seed=0,
)

utils.seed_everything(cfg.seed)
device = utils.get_device()
out_dir = utils.ensure_dir(cfg.output_dir)
print('Device:', device)
print('Output dir:', Path(out_dir).resolve())
print('Inpainting model:', cfg.inpaint_model_id)


In [ ]:
# Helpers for the notebook (multi-view input + merged point clouds)
def list_data_images(data_dir):
    exts = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}
    return sorted([p for p in Path(data_dir).glob('*') if p.is_file() and p.suffix in exts])

def merge_scene_point_cloud(scene, view_indices, conf_percentile, max_points=None, prefer_depth_unprojection=True, seed=0):
    pts_all, cols_all = [], []
    for idx in view_indices:
        pts, cols, _ = utils.build_point_cloud_from_scene(
            scene,
            view_idx=idx,
            conf_percentile=conf_percentile,
            max_points=None,
            prefer_depth_unprojection=prefer_depth_unprojection,
        )
        if len(pts):
            pts_all.append(pts)
            cols_all.append(cols)
    if not pts_all:
        return np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.float32)
    pts = np.concatenate(pts_all, axis=0).astype(np.float32)
    cols = np.concatenate(cols_all, axis=0).astype(np.float32)
    if max_points is not None and len(pts) > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(pts), size=max_points, replace=False)
        pts = pts[idx]
        cols = cols[idx]
    return pts, cols


In [ ]:
# Resolve the first 2 images from /data (minimum 2 images recommended for better geometry)
data_images = list_data_images(cfg.data_dir)
if len(data_images) < 2:
    raise RuntimeError(f'Need at least 2 images in {cfg.data_dir}. Found: {data_images}')

input_paths = [utils.ensure_png_copy(p) for p in data_images[:2]]
print('Using input images:')
for p in input_paths:
    print(' -', p)

for p in input_paths:
    display(Image.open(p).convert('RGB'))


In [ ]:
# 3) Download + run VGGT on the first 2 images
model = utils.load_vggt_model(cfg.vggt_model_id, device=device)
scene1 = utils.run_vggt_reconstruction(
    input_paths,
    model=model,
    device=device,
    preprocess_mode=cfg.preprocess_mode,
    model_id=cfg.vggt_model_id,
)

print(json.dumps(utils.describe_scene(scene1), indent=2, default=str))


In [ ]:
# 4) Plot the initial 3D reconstruction (merge points from both input views)
initial_view_indices = list(range(len(input_paths)))
pts1, cols1 = merge_scene_point_cloud(
    scene1,
    view_indices=initial_view_indices,
    conf_percentile=cfg.vggt_conf_percentile,
    max_points=cfg.max_points_plot,
    prefer_depth_unprojection=True,
    seed=cfg.seed,
)
print('Selected points for plotting:', len(pts1))

fig, ax = utils.plot_point_cloud_3d(
    pts1,
    cols1,
    title='Initial Multi-view VGGT 3D Reconstruction',
    point_size=0.2,
)
plt.show()


In [ ]:
# 4) Project the merged 3D model from the first recovered VGGT camera
pts_render, cols_render = merge_scene_point_cloud(
    scene1,
    view_indices=initial_view_indices,
    conf_percentile=cfg.vggt_conf_percentile,
    max_points=cfg.max_points_render,
    prefer_depth_unprojection=True,
    seed=cfg.seed,
)

base_extr = np.asarray(scene1['extrinsic'][0], dtype=np.float32)
base_intr = np.asarray(scene1['intrinsic'][0], dtype=np.float32)
base_hw = tuple(int(x) for x in scene1['image_hw'])

base_render = utils.render_projected_point_cloud(
    pts_render,
    cols_render,
    extrinsic=base_extr,
    intrinsic=base_intr,
    image_hw=base_hw,
    point_radius=cfg.render_point_radius,
)

preprocessed_input0 = Image.fromarray((np.clip(scene1['images_nhwc'][0], 0, 1) * 255).astype(np.uint8))
fig, axes = utils.plot_image_grid(
    [preprocessed_input0, base_render.image],
    titles=['VGGT preprocessed input (view 1)', 'Merged 3D projected from VGGT camera'],
    figsize=(12, 4),
)
plt.show()

print('Base projection valid ratio:', float(base_render.valid_mask.mean()))
print('Projected pixels:', base_render.projected_count)


In [ ]:
# 5) Move the camera a bit to the right (+ rotation), project into 2D, and show the result
novel_extr = utils.perturb_camera_extrinsic(
    base_extr,
    shift_right=cfg.novel_shift_right,
    yaw_deg=cfg.novel_yaw_deg,
    pitch_deg=cfg.novel_pitch_deg,
    roll_deg=cfg.novel_roll_deg,
)
novel_intr = base_intr.copy()

novel_render = utils.render_projected_point_cloud(
    pts_render,
    cols_render,
    extrinsic=novel_extr,
    intrinsic=novel_intr,
    image_hw=base_hw,
    point_radius=cfg.render_point_radius,
)

fig, axes = utils.plot_image_grid(
    [base_render.image, novel_render.image],
    titles=['Projection from original camera', 'Projection from shifted/rotated camera'],
    figsize=(12, 4),
)
plt.show()

print('Novel projection valid ratio:', float(novel_render.valid_mask.mean()))
print('Projected pixels:', novel_render.projected_count)


In [ ]:
# 6) Build a hole mask (regions not covered by the projected point cloud)
novel_img, hole_mask, hole_overlay = utils.prepare_novel_view_inpainting_inputs(
    novel_render,
    dilate_px=cfg.mask_dilate_px,
    close_px=cfg.mask_close_px,
)

hole_mask_ratio = float((np.asarray(hole_mask) > 0).mean())
print('Hole mask ratio:', hole_mask_ratio)

fig, axes = utils.plot_image_grid(
    [novel_img, hole_overlay, hole_mask.convert('RGB')],
    titles=['Novel-view projection', 'Hole mask overlay', 'Hole mask'],
    figsize=(15, 4),
)
plt.show()


In [ ]:
# 7) Local FLUX.2-klein-4B editing (preserve unmasked pixels exactly by mask compositing)
# Free VGGT weights before loading the inpainting pipeline to reduce memory pressure on Mac/MPS.
del model
utils.clear_loaded_model_caches()

inpaint_result = utils.inpaint_with_diffusion(
    image=novel_img,
    mask=hole_mask,
    prompt=cfg.inpaint_prompt,
    negative_prompt=cfg.inpaint_negative_prompt,
    model_id=cfg.inpaint_model_id,
    device=device,
    reference_images=input_paths,
    num_inference_steps=cfg.inpaint_steps,
    guidance_scale=cfg.inpaint_guidance_scale,
    seed=cfg.seed,
    allow_fallback_to_opencv=True,
)

# Verify pixels outside the mask are unchanged in the final composited image
orig_np = np.asarray(novel_img.convert('RGB'))
comp_np = np.asarray(inpaint_result.composited.convert('RGB'))
mask_np = np.asarray(hole_mask.convert('L')) > 0
changed_outside_mask = int(np.any(orig_np != comp_np, axis=-1)[~mask_np].sum())

print('Inpainting backend:', inpaint_result.backend)
print('Resized for model:', inpaint_result.resized_for_model)
print('Changed pixels outside mask (should be 0):', changed_outside_mask)

fig, axes = utils.plot_image_grid(
    [novel_img, inpaint_result.composited],
    titles=['Novel-view projection (with holes)', 'FLUX2-klein edited + masked composite'],
    figsize=(12, 4),
)
plt.show()


In [ ]:
# Save intermediate images
inpainted_path = Path(out_dir) / 'novel_view_inpainted_sdxl.png'
mask_path = Path(out_dir) / 'novel_view_hole_mask.png'
novel_path = Path(out_dir) / 'novel_view_projection.png'

utils.save_pil(inpaint_result.composited, inpainted_path)
utils.save_pil(hole_mask, mask_path)
utils.save_pil(novel_img, novel_path)

print('Saved:')
print(' -', inpainted_path.resolve())
print(' -', mask_path.resolve())
print(' -', novel_path.resolve())


In [ ]:
# 8) Run VGGT again with the 2 originals + the inpainted novel view
utils.clear_loaded_model_caches()
model2 = utils.load_vggt_model(cfg.vggt_model_id, device=device)
scene2 = utils.run_vggt_reconstruction(
    [*input_paths, inpainted_path],
    model=model2,
    device=device,
    preprocess_mode=cfg.preprocess_mode,
    model_id=cfg.vggt_model_id,
)

print(json.dumps(utils.describe_scene(scene2), indent=2, default=str))


In [ ]:
# Plot the augmented reconstruction (merge all views: 2 originals + inpainted novel view)
aug_view_indices = list(range(len(input_paths) + 1))
pts2, cols2 = merge_scene_point_cloud(
    scene2,
    view_indices=aug_view_indices,
    conf_percentile=cfg.vggt_conf_percentile,
    max_points=cfg.max_points_plot,
    prefer_depth_unprojection=True,
    seed=cfg.seed,
)
print('Selected points for plotting (augmented):', len(pts2))

fig, ax = utils.plot_point_cloud_3d(
    pts2,
    cols2,
    title='Augmented Multi-view VGGT Reconstruction (2 real + 1 inpainted)',
    point_size=0.2,
)
plt.show()


## Notes

- FLUX.2-klein local editing is heavier than the older SD1.5 inpainting model, but usually produces better texture/structure continuity.
- If quality is still weak, the biggest win is reducing the novel camera displacement so the hole mask gets smaller.
- A stronger next step is **geometry-based warping from both real views first**, then inpaint only the residual holes.
